# Lab 06: Deriving PCA from Linear Algebra

**Reference:** Goodfellow et al. *Deep Learning*, Chapter 2, Section 2.12

This capstone lab derives Principal Component Analysis (PCA) step by step, using
every major idea from Labs 1–5: matrix multiplication, orthogonality, eigendecomposition,
SVD, norms, and the trace operator. By the end you will understand not just *how* PCA works
but *why* each piece of the derivation is necessary.

---

### The Big Picture

We have data points in $\mathbb{R}^n$ and want a **lossy compression** to $\mathbb{R}^l$ with $l < n$.
We need two functions:

| Function | Notation | What it does |
|----------|----------|--------------|
| **Encoder** | $f(\mathbf{x}) = \mathbf{c}$ | Maps a data point to a lower-dimensional code |
| **Decoder** | $g(\mathbf{c}) \approx \mathbf{x}$ | Reconstructs the data point from the code |

PCA answers: *What is the best linear encoder/decoder pair?*

---

### Road map of the derivation (following the book)

1. **Constrain the decoder** to be a matrix multiply: $g(\mathbf{c}) = D\mathbf{c}$ where $D \in \mathbb{R}^{n \times l}$ with orthonormal columns.
2. **Derive the optimal encoder** given $D$: take the derivative of reconstruction error w.r.t. $\mathbf{c}$, set it to zero, and get $\mathbf{c}^* = D^\top \mathbf{x}$.
3. **Find the optimal $D$**: minimize $\mathbb{E}\left[\|\mathbf{x} - DD^\top \mathbf{x}\|_2^2\right]$. The book shows the columns of $D$ must be the eigenvectors of $X^\top X$ with the largest eigenvalues.
4. **Connect to SVD**: if $X = U\Sigma V^\top$, then $V$ gives the principal components directly.
5. **Applications**: variance explained, scree plots, whitening, and the link to autoencoders.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.set_printoptions(precision=4, suppress=True)

# ── Generate a 2D dataset with known covariance ──
rng = np.random.default_rng(42)
true_cov = np.array([[3.0, 2.0],
                     [2.0, 2.0]])
n = 300
X_raw = rng.multivariate_normal(mean=[1.0, 2.0], cov=true_cov, size=n)

# Center the data (subtract the mean) — essential for PCA
X = X_raw - X_raw.mean(axis=0)

print(f"Dataset shape : {X.shape}")
print(f"Sample mean   : {X.mean(axis=0)}  (should be ~0)")
print(f"True cov:\n{true_cov}")
print(f"Sample cov:\n{np.round(X.T @ X / (n - 1), 3)}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[:, 0], X[:, 1], alpha=0.4, s=15)
ax.set_aspect('equal')
ax.set_title('Centered 2D data')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 1: The Optimization Problem

*Book reference: Section 2.12, p. 45–47*

### Why centering?

Before PCA, we **always** subtract the mean from each feature. Why?

PCA finds directions of maximum *variance*. If the data has a nonzero mean, the
first "direction of maximum spread" would just point toward the mean — capturing
location rather than shape. Centering ensures $\mathbb{E}[\mathbf{x}] = \mathbf{0}$,
so the covariance matrix $\mathbb{E}[\mathbf{x}\mathbf{x}^\top]$ equals the second
moment matrix directly.

### The decoder constraint

The book constrains the decoder to
$$g(\mathbf{c}) = D\mathbf{c}, \qquad D \in \mathbb{R}^{n \times l},$$
where the columns of $D$ are **orthonormal**: $D^\top D = I_l$.

Why orthonormal?
- **Uniqueness**: Without a constraint, there are infinitely many equivalent solutions (you could
  scale one column of $D$ by 2 and divide the corresponding code entry by 2).
- **Interpretability**: Orthonormal columns mean the decoded point is a sum of
  unit-length, mutually perpendicular basis vectors weighted by the code entries.
- **Spectral theorem guarantee**: The covariance matrix is real symmetric, so by the
  spectral theorem (Lab 04) it has a full set of orthonormal eigenvectors. We want our
  decoder columns to *be* those eigenvectors.

### Deriving the optimal encoding

Given $D$ (fixed), we want the code $\mathbf{c}^*$ that minimizes reconstruction error for a single point:
$$\mathbf{c}^* = \arg\min_{\mathbf{c}} \|\mathbf{x} - D\mathbf{c}\|_2^2.$$

Expanding:
$$\|\mathbf{x} - D\mathbf{c}\|_2^2 = (\mathbf{x} - D\mathbf{c})^\top(\mathbf{x} - D\mathbf{c})
= \mathbf{x}^\top\mathbf{x} - 2\mathbf{x}^\top D\mathbf{c} + \mathbf{c}^\top \underbrace{D^\top D}_{= I_l} \mathbf{c}.$$

Taking the gradient w.r.t. $\mathbf{c}$ and setting to zero:
$$\nabla_{\mathbf{c}} = -2D^\top\mathbf{x} + 2\mathbf{c} = \mathbf{0}
\quad\Longrightarrow\quad \boxed{\mathbf{c}^* = D^\top \mathbf{x}}.$$

This is the **optimal encoder**: $f(\mathbf{x}) = D^\top \mathbf{x}$. The encoding is simply
the projection of $\mathbf{x}$ onto each column of $D$.

### Exercise 1 — Optimal Encoding Given D  *(basic)*

The book shows that for a fixed decoder matrix $D$ with orthonormal columns,
the optimal code is $\mathbf{c}^* = D^\top \mathbf{x}$ and the
reconstruction is $\hat{\mathbf{x}} = D D^\top \mathbf{x}$.

**Tasks:**
1. Pick a single row $\mathbf{x}$ from the data.
2. Choose an arbitrary unit vector $\mathbf{d}$ as a 1-component decoder ($l = 1$, so $D$ is a column vector).
3. Compute the optimal code $c^* = \mathbf{d}^\top \mathbf{x}$ (a scalar).
4. Compute the reconstruction $\hat{\mathbf{x}} = c^* \mathbf{d}$.
5. Verify that $\hat{\mathbf{x}} - \mathbf{x}$ is orthogonal to $\mathbf{d}$ (the residual is perpendicular to the subspace).

In [ ]:
# Exercise 1: Optimal encoding given D
# d=2 (ambient), k=1 (code dimension)

# Pick one data point
x_point = X[0]   # shape (2,)

# Choose an arbitrary unit-length decoder direction
d_vec = np.array([3.0, 4.0])
d_vec = d_vec / np.linalg.norm(d_vec)   # unit vector

# (a) Optimal code: c* = d^T x  (scalar)
c_star = d_vec @ x_point

# (b) Reconstruction: x_hat = c* * d
x_hat = c_star * d_vec

# (c) Residual: r = x_hat - x  (should be perpendicular to d)
residual = x_hat - x_point
dot_residual_d = residual @ d_vec   # should be ~0


# ── Verification ──
assert abs(np.linalg.norm(d_vec) - 1.0) < 1e-10, "d must be unit length"
assert abs(c_star - d_vec @ x_point) < 1e-10, "c_star incorrect"
assert np.allclose(x_hat, c_star * d_vec), "x_hat incorrect"
assert abs(dot_residual_d) < 1e-10, \
    f"Residual not orthogonal to d: dot = {dot_residual_d:.2e}"

print(f"x          = {x_point}")
print(f"d          = {d_vec}")
print(f"c*         = {c_star:.4f}")
print(f"x_hat      = {x_hat}")
print(f"residual   = {residual}")
print(f"r . d      = {dot_residual_d:.2e}  (orthogonal!)")
print(f"||x - x_hat|| = {np.linalg.norm(residual):.4f}")
print("\nExercise 1: PASSED")

### Exercise 2 — Sample Covariance Matrix  *(basic)*

The book (eq. 2.59) shows the optimal $D$ depends on the matrix $X^\top X$
(which is proportional to the sample covariance for centered data).

**Key insight:** For centered data the sample covariance is
$$C = \frac{1}{n-1} X^\top X.$$

This matrix is:
- **Symmetric** ($C = C^\top$), because $(X^\top X)^\top = X^\top X$.
- **Positive semi-definite** (all eigenvalues $\geq 0$), because for any $\mathbf{v}$,
  $\mathbf{v}^\top X^\top X \mathbf{v} = \|X\mathbf{v}\|^2 \geq 0$.

These two properties are exactly what the **spectral theorem** (Lab 04) requires:
$C$ can be diagonalized as $C = Q\Lambda Q^\top$ with real eigenvalues and
orthonormal eigenvectors. This is the mathematical foundation of PCA.

**Tasks:**
1. Compute the sample covariance matrix `C = X.T @ X / (n-1)`.  Shape: `(2, 2)`.
2. Verify it is symmetric: `C == C.T`.
3. Verify it is PSD: all eigenvalues $\geq 0$.

In [ ]:
# Exercise 2: Sample covariance matrix

# (a) Compute C — shape (2, 2)
C = X.T @ X / (n - 1)

# (b) Is it symmetric? Compute max absolute difference between C and C.T
symmetry_error = np.max(np.abs(C - C.T))   # scalar

# (c) Compute eigenvalues of C
eigenvalues_C = np.linalg.eigvalsh(C)    # 1D array of length 2


# ── Verification ──
assert C is not None,             "C not implemented"
assert symmetry_error is not None, "symmetry_error not implemented"
assert eigenvalues_C is not None,  "eigenvalues_C not implemented"
assert C.shape == (2, 2),          f"C shape: expected (2,2), got {C.shape}"
assert symmetry_error < 1e-12,     f"C is not symmetric! max diff = {symmetry_error:.2e}"
assert np.all(eigenvalues_C >= -1e-10), \
    f"C has negative eigenvalue: {eigenvalues_C.min():.4f}  — not PSD!"

print(f"Sample covariance C:\n{np.round(C, 4)}")
print(f"True covariance:\n{true_cov}")
print(f"Symmetry error     : {symmetry_error:.2e}")
print(f"Eigenvalues of C   : {np.round(eigenvalues_C, 4)}  (all >= 0 -> PSD confirmed)")
print("\nExercise 2: PASSED")

---
## Part 2: Finding Principal Directions

*Book reference: Section 2.12, p. 47–48*

### The optimization for $D$

Substituting $\mathbf{c}^* = D^\top \mathbf{x}$ back into the reconstruction, the
per-point reconstruction is $\hat{\mathbf{x}} = DD^\top\mathbf{x}$. The matrix
$P = DD^\top$ is an **orthogonal projection** onto the column space of $D$
(this is the projection matrix from Lab 02).

We want to minimize the expected reconstruction error over the dataset:
$$\min_D \; \frac{1}{n}\sum_{i=1}^{n} \|\mathbf{x}^{(i)} - DD^\top\mathbf{x}^{(i)}\|_2^2
= \min_D \; \frac{1}{n}\|X - XDD^\top\|_F^2.$$

The book shows (using the trace operator from Lab 05) that this is equivalent to:
$$\max_D \; \operatorname{Tr}(D^\top X^\top X D) \quad \text{subject to} \quad D^\top D = I_l.$$

**Why trace?** The Frobenius norm squared equals the trace of $A^\top A$ (Lab 05).
After expanding $\|X - XDD^\top\|_F^2$ and dropping terms that don't depend on $D$,
the remaining term is $-\operatorname{Tr}(D^\top X^\top X D)$. Minimizing a negative is
maximizing the positive.

### The solution

By the **Rayleigh quotient** characterization of eigenvalues, the constrained
maximum is achieved when the columns of $D$ are the eigenvectors of $X^\top X$
corresponding to the $l$ **largest** eigenvalues.

Equivalently, the columns of $D$ are the top eigenvectors of the covariance matrix $C = X^\top X / (n-1)$,
since dividing by a positive constant doesn't change the eigenvectors.

### Intuition: directions of maximum variance

Consider projecting data onto a unit vector $\mathbf{d}$. The projected values are
$z_i = \mathbf{d}^\top \mathbf{x}^{(i)}$, and their variance is:
$$\operatorname{Var}(z) = \frac{1}{n-1}\sum_i (\mathbf{d}^\top \mathbf{x}^{(i)})^2
= \mathbf{d}^\top \left(\frac{X^\top X}{n-1}\right) \mathbf{d}
= \mathbf{d}^\top C \mathbf{d}.$$

Maximizing $\mathbf{d}^\top C \mathbf{d}$ subject to $\|\mathbf{d}\| = 1$ is the
**Rayleigh quotient** problem. The maximum is achieved at the eigenvector with
the largest eigenvalue. So the first principal component is literally the direction
of maximum variance.

### Exercise 3 — Variance Maximization  *(intermediate)*

The first principal component maximizes variance of the projected data.

**Tasks:**
1. Compute the top eigenvector $\mathbf{d}_1$ of $X^\top X$ (the one with the largest eigenvalue).
2. Sweep 100 unit vectors at angles $\theta \in [0, \pi)$ and compute the variance of $X\mathbf{d}$ for each.
3. Verify that the eigenvector direction gives the maximum variance.
4. Plot the variance as a function of $\theta$ and mark the eigenvector angle.

In [ ]:
# Exercise 3: Variance maximization — two ways

XtX = X.T @ X   # (2, 2) — proportional to covariance

# (a) Eigenvector with largest eigenvalue
evals3, evecs3 = np.linalg.eigh(XtX)
d1 = evecs3[:, np.argmax(evals3)]   # shape (2,)
var_d1 = np.var(X @ d1, ddof=1)     # projected variance along d1

# (b) Brute-force sweep over 100 angles in [0, pi)
thetas = np.linspace(0, np.pi, 100, endpoint=False)
variances = np.array([
    np.var(X @ np.array([np.cos(t), np.sin(t)]), ddof=1)
    for t in thetas
])
best_idx = np.argmax(variances)
best_theta = thetas[best_idx]
var_brute = variances[best_idx]

# Angle of the eigenvector
theta_eig = np.arctan2(d1[1], d1[0]) % np.pi


# ── Verification ──
assert abs(var_d1 - var_brute) / var_d1 < 0.02, \
    f"Eigenvector variance {var_d1:.4f} != brute force {var_brute:.4f}"
assert abs(theta_eig - best_theta) < 0.05, \
    f"Angles differ: eig={theta_eig:.4f}, brute={best_theta:.4f}"

print(f"d1 (top eigvec) = {d1}")
print(f"Variance along d1            = {var_d1:.4f}")
print(f"Max variance (brute force)   = {var_brute:.4f}")
print(f"Eigenvector angle            = {np.degrees(theta_eig):.1f} deg")
print(f"Brute-force best angle       = {np.degrees(best_theta):.1f} deg")
print("\nExercise 3: PASSED")

# ── Plot variance vs. angle ──
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(np.degrees(thetas), variances, 'b-', lw=1.5, label='Projected variance')
ax.axvline(np.degrees(theta_eig), color='red', ls='--', lw=1.5,
           label=f'Top eigenvector ({np.degrees(theta_eig):.1f}$^\\circ$)')
ax.set_xlabel('Projection angle (degrees)')
ax.set_ylabel('Variance of $Xd$')
ax.set_title('PCA = direction of maximum variance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 4 — Greedy (Sequential) PCA vs. Batch Eigendecomposition  *(intermediate)*

The book describes PCA as finding components one at a time: the $k$-th
component maximizes remaining variance while being orthogonal to the first $k-1$.

**Why orthogonal?** The covariance matrix is symmetric, so the spectral theorem
guarantees that its eigenvectors are orthogonal. Orthogonal components capture
**uncorrelated** information — each component adds genuinely new information
rather than partially redundant information. This is not an arbitrary choice;
it falls directly out of the optimization.

We can verify this in two ways:
- **Batch**: eigendecompose $X^\top X$, take the top $l$ eigenvectors.
- **Greedy (deflation)**: find the top eigenvector, subtract its contribution from the data, repeat.

Both should give the same result.

**Tasks:**
1. Compute all eigenvectors of $X^\top X$ sorted by decreasing eigenvalue (batch).
2. Find $\mathbf{d}_1$ (top eigenvector), deflate the data: $X_{\text{resid}} = X - X\mathbf{d}_1\mathbf{d}_1^\top$, then find $\mathbf{d}_2$.
3. Compare the greedy and batch results (they should match up to sign).

In [ ]:
# Exercise 4: Greedy PCA vs. eigendecomposition

# --- Batch eigendecomposition (reference) ---
# Sort eigenvalues DESCENDING; eigenvectors as columns of a (2,2) matrix
evals_batch, evecs_batch = np.linalg.eigh(X.T @ X)
# eigh returns ascending order — reverse to get descending
idx_desc = np.argsort(evals_batch)[::-1]
evecs_desc = evecs_batch[:, idx_desc]
d1_ref = evecs_desc[:, 0]   # shape (2,) — largest eigenvalue's eigenvector
d2_ref = evecs_desc[:, 1]   # shape (2,) — second eigenvector


# --- Greedy sequential approach ---
# Step 1: d1_greedy = eigenvector of X.T @ X with largest eigenvalue
evals_g, evecs_g = np.linalg.eigh(X.T @ X)
d1_greedy = evecs_g[:, np.argmax(evals_g)]   # shape (2,)

# Step 2: Deflate — project d1 out of the data
# x_resid = x - (x . d1) d1  for each row
X_resid = X - np.outer(X @ d1_greedy, d1_greedy)  # shape (n, 2)

# Step 3: d2_greedy = eigenvector of X_resid.T @ X_resid with largest eigenvalue
evals_g2, evecs_g2 = np.linalg.eigh(X_resid.T @ X_resid)
d2_greedy = evecs_g2[:, np.argmax(evals_g2)]   # shape (2,)


# ── Verification ──
for name, g, r in [('d1', d1_greedy, d1_ref), ('d2', d2_greedy, d2_ref)]:
    assert g is not None, f"{name}_greedy not implemented"
    assert r is not None, f"{name}_ref not implemented"
    align = abs(g @ r)
    assert align > 0.999, f"{name}: greedy and batch disagree, |cos angle| = {align:.6f}"
    print(f"{name} greedy: {np.round(g,4)},  ref: {np.round(r,4)},  |cos theta|={align:.6f}")

print("\nExercise 4: PASSED")
print(f"d1 . d2 = {d1_greedy @ d2_greedy:.2e}  (should be ~0 — principal components are orthogonal)")

### Exercise 5 — Reconstruction Error Analysis  *(intermediate)*

The book (eq. 2.61) shows that the minimum reconstruction error equals
the **sum of the discarded eigenvalues**:

$$\text{Reconstruction Error} = \sum_{j=l+1}^{n} \lambda_j,$$

where $\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_n$ are eigenvalues of $X^\top X$
and we keep $l$ components.

**Why?** The total variance is $\sum_j \lambda_j$. Keeping $l$ components captures
$\sum_{j=1}^{l} \lambda_j$. The difference — the discarded eigenvalues — is exactly
the reconstruction error. This is the **Eckart-Young theorem** applied to the data matrix:
the best rank-$l$ approximation (in Frobenius norm) drops the smallest singular values.

**Variance explained.** Each component's contribution is:
$$\text{fraction}_k = \frac{\lambda_k}{\sum_j \lambda_j}.$$

The **scree plot** shows eigenvalue magnitude vs. component index. Look for the **elbow** —
the point where eigenvalues drop sharply, indicating that subsequent components
capture mostly noise rather than signal.

**Tasks:**
1. For $l = 0, 1, 2$ (keep 0, 1, or 2 components), compute reconstruction error directly.
2. Verify it matches the sum of discarded eigenvalues.
3. Compute the **variance explained** ratio for each component.

In [ ]:
# Exercise 5: Reconstruction error vs. number of components

# Get all eigenvalues of X.T @ X in descending order
evals_all = np.sort(np.linalg.eigvalsh(X.T @ X))[::-1]   # shape (2,)

# Build the decoder D whose columns are the top eigenvectors (descending order)
_, evecs_all = np.linalg.eigh(X.T @ X)
D_full = evecs_all[:, np.argsort(np.linalg.eigvalsh(X.T @ X))[::-1]]  # (2, 2)

errors_direct = []    # reconstruction errors computed directly
errors_formula = []   # reconstruction errors from sum of discarded eigenvalues

for l in range(3):  # l = 0, 1, 2
    if l == 0:
        X_recon = np.zeros_like(X)
    else:
        D_l = D_full[:, :l]                    # first l columns
        X_recon = X @ D_l @ D_l.T              # project and reconstruct
    err_direct = np.sum((X - X_recon) ** 2)    # total squared error
    err_formula = np.sum(evals_all[l:])        # sum of discarded eigenvalues
    errors_direct.append(err_direct)
    errors_formula.append(err_formula)

# Variance explained ratio
var_explained = evals_all / np.sum(evals_all)
cumulative_var = np.cumsum(var_explained)


# ── Verification ──
for l in range(3):
    assert abs(errors_direct[l] - errors_formula[l]) < 1e-6, \
        f"l={l}: direct={errors_direct[l]:.4f} != formula={errors_formula[l]:.4f}"
    print(f"l={l}: error(direct)={errors_direct[l]:10.2f},  "
          f"error(eigenvalues)={errors_formula[l]:10.2f}")

print(f"\nEigenvalues of X.T @ X : {np.round(evals_all, 2)}")
print(f"Variance explained     : {np.round(var_explained * 100, 2)}%")
print(f"Cumulative var explained: {np.round(cumulative_var * 100, 2)}%")
print("\nExercise 5: PASSED")

---
## Part 3: PCA via SVD

*Book reference: Section 2.8 (SVD) and Section 2.12, p. 48–49*

### The connection

In practice, PCA is almost always computed via the SVD rather than an explicit
eigendecomposition of the covariance matrix. Here is why:

If $X = U\Sigma V^\top$ is the (economy) SVD of the centered data matrix, then:

$$X^\top X = (U\Sigma V^\top)^\top (U\Sigma V^\top) = V\Sigma^\top U^\top U \Sigma V^\top
= V \Sigma^2 V^\top.$$

Comparing with the eigendecomposition $X^\top X = Q\Lambda Q^\top$, we see:
- **$V$ = matrix of eigenvectors** (the principal component directions)
- **$\Sigma^2$ = diagonal matrix of eigenvalues** of $X^\top X$
- The eigenvalues of the covariance matrix $C = X^\top X / (n-1)$ are $\sigma_i^2 / (n-1)$

The **projected data** (scores) can be computed as:
$$Z = XV = U\Sigma$$

This is numerically more stable than forming $X^\top X$ explicitly (which squares
the condition number) and is the standard implementation in scikit-learn.

### Exercise 6 — PCA via SVD  *(intermediate)*

**Tasks:**
1. Compute the full SVD of $X$: $X = U\Sigma V^\top$.
2. Verify that the rows of $V^\top$ (= columns of $V$) match the eigenvectors of $X^\top X$.
3. Compute projected scores two ways: $U\Sigma$ and $XV$. Verify they match.
4. Verify the eigenvalue relationship: eigenvalues of covariance $= \sigma_i^2 / (n-1)$.

In [ ]:
# Exercise 6: PCA via SVD

# (a) Compute economy SVD
U, s, Vt = np.linalg.svd(X, full_matrices=False)
# U: (n, d), s: (d,), Vt: (d, d)  where d = 2

# (b) Principal components from SVD vs. eigendecomposition of X.T @ X
# pc_svd: rows of Vt — already in descending singular value order
pc_svd = Vt   # shape (d, d)

# Eigendecomposition of X.T @ X for reference (sort descending)
evals_ref, evecs_ref_raw = np.linalg.eigh(X.T @ X)
idx6 = np.argsort(evals_ref)[::-1]
evecs_ref = evecs_ref_raw[:, idx6]   # columns sorted descending
evals_ref = evals_ref[idx6]

# (c) Projected scores two ways
scores_svd   = U @ np.diag(s)   # shape (n, d)
scores_eigen = X @ Vt.T         # X @ V where V = Vt.T, shape (n, d)

# (d) Eigenvalues of covariance from SVD vs. direct
lambdas_svd  = s**2 / (n - 1)         # shape (d,)
lambdas_cov  = evals_ref / (n - 1)    # eigenvalues of X.T@X / (n-1)


# ── Verification ──
assert U is not None,   "U not implemented"
assert s is not None,   "s not implemented"
assert Vt is not None,  "Vt not implemented"

# Check shapes
assert U.shape == (n, 2),   f"U shape {U.shape}"
assert s.shape == (2,),     f"s shape {s.shape}"
assert Vt.shape == (2, 2),  f"Vt shape {Vt.shape}"

# Check principal components match (up to sign)
for i, (pc, ev) in enumerate(zip(pc_svd, evecs_ref.T)):
    align = abs(pc @ ev)
    assert align > 0.999, f"PC {i}: |cos theta| = {align:.6f} — SVD and eig disagree"
    print(f"PC {i+1}: SVD vs eig |cos theta| = {align:.6f}")

# Check scores match (up to global sign)
align_scores = np.corrcoef(scores_svd[:, 0], scores_eigen[:, 0])[0, 1]
assert abs(align_scores) > 0.9999, f"Scores don't match: corr = {align_scores:.6f}"
print(f"Score correlation (SVD vs X@V): {align_scores:.6f}")

# Check eigenvalue relationship
assert np.allclose(lambdas_svd, lambdas_cov, atol=1e-8), \
    f"Eigenvalue mismatch: SVD={lambdas_svd}, cov={lambdas_cov}"
print(f"Cov eigenvalues (from SVD)  : {np.round(lambdas_svd, 4)}")
print(f"Cov eigenvalues (direct)    : {np.round(lambdas_cov, 4)}")
print("\nExercise 6: PASSED")

### Exercise 7 — PCA on High-Dimensional Data  *(intermediate)*

Now let us work in a more realistic setting: 50-dimensional data where the
"true" signal lives in a much lower-dimensional subspace.

This demonstrates the real power of PCA: **discovering the intrinsic
dimensionality** of data. If the first few eigenvalues dominate, the data
effectively lives in a low-dimensional subspace despite being embedded
in high-dimensional space.

**Tasks:**
1. Generate 50D data with only 5 "true" dimensions (plus noise).
2. Compute PCA via SVD.
3. Create a **scree plot** showing eigenvalues vs. component index.
4. Create a **cumulative variance** plot.
5. Find $l^*$ = smallest $l$ that captures $\geq 95\%$ of variance.

In [ ]:
# Exercise 7: PCA on high-dimensional data

np.random.seed(7)
n_high, d_high, d_true = 200, 50, 5

# Generate data: 5 true dimensions embedded in 50D via a random projection + noise
Z_true = np.random.randn(n_high, d_true) * np.array([10, 6, 4, 2, 1])  # varying scales
A_proj = np.random.randn(d_true, d_high)    # random embedding
noise  = np.random.randn(n_high, d_high) * 0.5
X_high = Z_true @ A_proj + noise
X_high = X_high - X_high.mean(axis=0)       # center

# PCA via SVD
U_h, s_h, Vt_h = np.linalg.svd(X_high, full_matrices=False)
lambdas_h = s_h**2 / (n_high - 1)            # covariance eigenvalues
var_explained_h = lambdas_h / lambdas_h.sum()
cum_var_h = np.cumsum(var_explained_h)

# Find l* — smallest l capturing >= 95% variance
l_star = int(np.searchsorted(cum_var_h, 0.95) + 1)


# ── Verification ──
assert lambdas_h is not None, "lambdas not computed"
assert l_star <= d_true + 2, \
    f"l* = {l_star} seems too large for {d_true} true dimensions"

print(f"Top 10 eigenvalues: {np.round(lambdas_h[:10], 2)}")
print(f"Cumulative variance (first 10): {np.round(cum_var_h[:10] * 100, 1)}%")
print(f"l* (95% variance) = {l_star}  (true dimensionality = {d_true})")
print("\nExercise 7: PASSED")

# ── Scree plot and cumulative variance ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
ax1.bar(range(1, 16), lambdas_h[:15], color='steelblue', alpha=0.7)
ax1.axvline(l_star + 0.5, color='red', ls='--', label=f'l* = {l_star}')
ax1.set_xlabel('Component')
ax1.set_ylabel('Eigenvalue ($\\sigma_i^2 / (n-1)$)')
ax1.set_title('Scree Plot — look for the elbow')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative variance
ax2.plot(range(1, d_high + 1), cum_var_h * 100, 'o-', ms=3)
ax2.axhline(95, color='red', ls='--', label='95% threshold')
ax2.axvline(l_star, color='red', ls=':', alpha=0.5)
ax2.set_xlabel('Number of components ($l$)')
ax2.set_ylabel('Cumulative variance explained (%)')
ax2.set_title('Cumulative Variance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 4: Connecting Everything

*This section ties together eigendecomposition, SVD, projection, and reconstruction
from Labs 1–5 into a single algorithm built from scratch.*

### The full PCA pipeline

Let us summarize the complete algorithm:

1. **Center** the data: $X \leftarrow X - \bar{X}$
2. **Compute** $X^\top X$ (or use SVD directly)
3. **Find** the top $l$ eigenvectors of $X^\top X$ (these become the columns of $D$)
4. **Encode**: $C = XD$ (project data onto principal components)
5. **Decode**: $\hat{X} = CD^\top = XDD^\top$ (reconstruct from code)
6. **Reconstruction error** $= \sum_{j=l+1}^n \lambda_j$ (sum of discarded eigenvalues)

### Exercise 8 — PCA from Primitives (Power Iteration + Gram-Schmidt)  *(challenging)*

Build PCA using only:
- **Power iteration** (Lab 04) to find eigenvectors one at a time
- **Gram-Schmidt** (Lab 01) to ensure orthogonality
- **Matrix deflation** to remove found components

This shows that PCA is truly built from the linear algebra primitives
we have been studying.

**Tasks:**
1. Implement `power_iteration(A)` — find the dominant eigenvector of a symmetric PSD matrix.
2. Implement `gram_schmidt(vectors)` — orthonormalize a list of vectors.
3. Implement `pca_primitives(X, k)` — full PCA using deflation.
4. Compare with `np.linalg.eigh` results.

In [ ]:
# Exercise 8: PCA from primitives

def power_iteration(A, num_iterations=1000, tol=1e-10, seed=0):
    """Find the dominant eigenvector of symmetric PSD matrix A.
    Returns: (eigenvalue, eigenvector)
    """
    rng_pi = np.random.default_rng(seed)
    d = A.shape[0]
    # Initialize v as a random unit vector
    v = rng_pi.standard_normal(d)
    v = v / np.linalg.norm(v)

    for _ in range(num_iterations):
        v_new = A @ v                         # A @ v
        norm = np.linalg.norm(v_new)          # ||A @ v||
        v_new = v_new / norm                  # normalised

        # Convergence check: vectors agree up to sign
        if np.linalg.norm(v_new - v) < tol or np.linalg.norm(v_new + v) < tol:
            break
        v = v_new

    # Rayleigh quotient for eigenvalue
    lam = v @ A @ v   # v.T @ A @ v
    return lam, v


def gram_schmidt(vectors):
    """Orthonormalize a list of vectors using Gram-Schmidt.
    Args: vectors — list of 1D numpy arrays, each shape (d,)
    Returns: list of orthonormal vectors
    """
    basis = []
    for v in vectors:
        # Subtract projections onto all previously found basis vectors
        w = v.copy()
        for u in basis:
            w = w - (w @ u) * u   # w - (w . u) u
        norm = np.linalg.norm(w)  # ||w||
        if norm > 1e-10:          # avoid degenerate directions
            basis.append(w / norm)   # w / norm
    return basis


def pca_primitives(X, k):
    """PCA via power iteration + deflation.
    Returns: (eigenvalues, eigenvectors_as_columns)
    """
    A = X.T @ X
    evals_list = []
    evecs_list = []

    for i in range(k):
        lam, v = power_iteration(A, seed=i)
        evals_list.append(lam)
        evecs_list.append(v)
        # Deflate: remove this component from A
        A = A - lam * np.outer(v, v)

    # Gram-Schmidt to clean up any numerical drift
    evecs_clean = gram_schmidt(evecs_list)

    return np.array(evals_list), np.column_stack(evecs_clean)


# --- Test on our 2D data ---
evals_prim, evecs_prim = pca_primitives(X, 2)

# Reference from numpy
evals_np, evecs_np_raw = np.linalg.eigh(X.T @ X)
idx_np = np.argsort(evals_np)[::-1]
evals_np = evals_np[idx_np]
evecs_np = evecs_np_raw[:, idx_np]


# ── Verification ──
# Eigenvalues should match
assert np.allclose(evals_prim, evals_np, rtol=1e-4), \
    f"Eigenvalues: primitive={evals_prim} vs numpy={evals_np}"

# Eigenvectors should match up to sign
for i in range(2):
    align = abs(evecs_prim[:, i] @ evecs_np[:, i])
    assert align > 0.999, f"PC {i+1}: |cos theta| = {align:.6f}"
    print(f"  PC {i+1}: {align:.6f}")

print("\nExercise 8: PASSED")

### Exercise 9 — ZCA Whitening  *(challenging)*

**Whitening** transforms data so that its covariance becomes the identity matrix $I$.
The whitened data has:
- Unit variance along every direction
- Zero correlation between all pairs of features

There are two common forms:

| Type | Formula | Result |
|------|---------|--------|
| **PCA whitening** | $W_{\text{PCA}} = \Lambda^{-1/2} Q^\top$ | Data is rotated into PC space, then scaled |
| **ZCA whitening** | $W_{\text{ZCA}} = Q \Lambda^{-1/2} Q^\top$ | Data stays in original space, just "sphered" |

where $C = Q\Lambda Q^\top$ is the eigendecomposition of the covariance.

**Why whitening matters in ML:**
- Makes gradient descent converge faster (the loss surface becomes more spherical)
- Required preprocessing for some algorithms (ICA, some forms of sparse coding)
- ZCA whitening keeps data "closest" to the original (minimizes $\|WX^\top - X^\top\|_F$)

**Tasks:**
1. Compute both PCA whitening and ZCA whitening matrices.
2. Apply them to the data.
3. Verify that the covariance of the whitened data is approximately $I$.

In [ ]:
# Exercise 9: ZCA whitening

# Sample covariance (with bias correction)
C_zca = X.T @ X / (n - 1)

# Eigendecomposition of the covariance
eigvals_zca, Q_zca = np.linalg.eigh(C_zca)

# Lambda^{-1/2} (diagonal matrix of inverse square roots of eigenvalues)
# Add small epsilon for numerical stability
eps = 1e-10
Lambda_inv_sqrt = np.diag(1.0 / np.sqrt(eigvals_zca + eps))

# PCA whitening: W_pca = Lambda^{-1/2} @ Q^T
W_pca = Lambda_inv_sqrt @ Q_zca.T
X_pca_white = X @ W_pca.T

# ZCA whitening: W_zca = Q @ Lambda^{-1/2} @ Q^T
W_zca = Q_zca @ Lambda_inv_sqrt @ Q_zca.T
X_zca_white = X @ W_zca.T

# Verify covariance is approximately I
C_pca_white = X_pca_white.T @ X_pca_white / (n - 1)
C_zca_white = X_zca_white.T @ X_zca_white / (n - 1)


# ── Verification ──
I2 = np.eye(2)
assert np.allclose(C_pca_white, I2, atol=1e-6), \
    f"PCA whitening failed:\n{C_pca_white}"
assert np.allclose(C_zca_white, I2, atol=1e-6), \
    f"ZCA whitening failed:\n{C_zca_white}"

print("Covariance after PCA whitening:")
print(np.round(C_pca_white, 6))
print("\nCovariance after ZCA whitening:")
print(np.round(C_zca_white, 6))
print("\nExercise 9: PASSED")

# ── Visualize whitening ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, data, title in zip(axes,
        [X, X_pca_white, X_zca_white],
        ['Original (centered)', 'PCA whitened', 'ZCA whitened']):
    ax.scatter(data[:, 0], data[:, 1], alpha=0.4, s=15)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    # Draw unit circle for reference
    theta_circ = np.linspace(0, 2 * np.pi, 100)
    r = 2.0
    ax.plot(r * np.cos(theta_circ), r * np.sin(theta_circ),
            'r--', alpha=0.3, lw=1)
plt.tight_layout()
plt.show()

---
## Part 5: Visualization and the Reconstruction Story

Let us see PCA in action: projecting, reconstructing, and visualizing
what information is kept vs. lost.

### Exercise 10 — Projection and Reconstruction Visualization  *(intermediate)*

**Tasks:**
1. Compute the principal components of the 2D data.
2. Project onto the first PC only ($l = 1$).
3. Reconstruct: $\hat{\mathbf{x}} = (\mathbf{x}^\top \mathbf{d}_1) \mathbf{d}_1$.
4. Plot the original data, the PC direction, and the reconstruction (showing the "collapse" onto the line).

In [ ]:
# Exercise 10: Projection and reconstruction visualization

# Principal components via SVD
U10, s10, Vt10 = np.linalg.svd(X, full_matrices=False)
pc1 = Vt10[0]  # first principal component direction

# Project onto PC1 and reconstruct
scores_1d = X @ pc1               # (n,) — scalar projections
X_recon_1d = np.outer(scores_1d, pc1)  # (n, 2) — reconstructed points

# Reconstruction error
recon_err = np.mean(np.sum((X - X_recon_1d)**2, axis=1))
total_var = np.sum(s10**2) / (n - 1)
var_kept = s10[0]**2 / (n - 1)

print(f"PC1 direction: {pc1}")
print(f"Variance kept (PC1): {var_kept:.4f} / {total_var:.4f} = {var_kept/total_var*100:.1f}%")
print(f"Mean reconstruction error: {recon_err:.4f}")

# ── Visualization ──
fig, ax = plt.subplots(figsize=(8, 7))

# Original data
ax.scatter(X[:, 0], X[:, 1], alpha=0.3, s=15, color='steelblue', label='Original')

# Reconstructed data (on the PC1 line)
ax.scatter(X_recon_1d[:, 0], X_recon_1d[:, 1], alpha=0.4, s=15,
           color='red', label='Reconstruction (l=1)', zorder=3)

# Draw lines from original to reconstruction (residuals)
for i in range(0, n, 5):  # every 5th point for clarity
    ax.plot([X[i, 0], X_recon_1d[i, 0]], [X[i, 1], X_recon_1d[i, 1]],
            'gray', alpha=0.3, lw=0.5)

# Draw PC directions
scale = 4
ax.annotate('', xy=scale*pc1, xytext=-scale*pc1,
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(scale*pc1[0]+0.2, scale*pc1[1]+0.2, 'PC1', fontsize=12,
        color='red', fontweight='bold')

pc2 = Vt10[1]
scale2 = 2
ax.annotate('', xy=scale2*pc2, xytext=-scale2*pc2,
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(scale2*pc2[0]+0.2, scale2*pc2[1]+0.2, 'PC2', fontsize=12,
        color='green', fontweight='bold')

ax.set_aspect('equal')
ax.set_title('PCA: Projection onto PC1 and Reconstruction\n'
             f'Variance explained: {var_kept/total_var*100:.1f}%')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nExercise 10: PASSED")

---
## Part 6: ML Connections

### Why PCA matters for machine learning

PCA is not just a textbook exercise — it is one of the most widely used
techniques in practice:

| Application | How PCA helps |
|-------------|---------------|
| **Dimensionality reduction** | Reduce features before training to avoid the curse of dimensionality |
| **Feature visualization** | Project high-dimensional data to 2D or 3D for plotting |
| **Noise reduction** | Discard components with small eigenvalues (mostly noise) |
| **Decorrelation** | PCA-transformed features are uncorrelated, simplifying models |
| **Compression** | Store only the top $l$ scores instead of $n$ features per data point |

### The autoencoder connection

A **linear autoencoder** has:
- Encoder: $\mathbf{c} = W_e \mathbf{x} + \mathbf{b}_e$
- Decoder: $\hat{\mathbf{x}} = W_d \mathbf{c} + \mathbf{b}_d$

When trained with MSE loss and no nonlinearities, the weights converge to
exactly the PCA subspace (Baldi & Hornik, 1989). Specifically:
- The decoder weights $W_d$ span the same subspace as the top $l$ eigenvectors
- The encoder learns $W_e = W_d^\top$ (the pseudo-inverse for orthogonal $W_d$)

**Deep (nonlinear) autoencoders** generalize this: they can learn *curved*
manifolds rather than flat subspaces. But the linear case = PCA is the
foundation.

### Other generalizations

- **Kernel PCA**: Apply PCA in a feature space induced by a kernel function.
  Can capture nonlinear structure.
- **Probabilistic PCA**: A generative model where the latent space is Gaussian.
  Maximum likelihood estimation gives the same principal components.
- **Sparse PCA**: Add an L1 penalty to the loadings for interpretable components.
- **Incremental PCA**: Process data in mini-batches (for datasets too large for memory).

### Exercise 11 — Linear Autoencoder = PCA  *(challenging)*

Train a simple linear autoencoder (no activation functions) with gradient
descent and verify it converges to the PCA subspace.

The autoencoder has:
- Encoder: $\mathbf{c} = W_e \mathbf{x}$ where $W_e \in \mathbb{R}^{l \times n}$
- Decoder: $\hat{\mathbf{x}} = W_d \mathbf{c}$ where $W_d \in \mathbb{R}^{n \times l}$
- Loss: $\frac{1}{n}\sum_i \|\mathbf{x}_i - W_d W_e \mathbf{x}_i\|^2$

After training, the columns of $W_d$ should span the same subspace as the top $l$
principal components. We verify by checking that the projection matrix
$W_d W_e \approx DD^\top$ (up to rotation within the subspace).

**Tasks:**
1. Initialize $W_e$ and $W_d$ randomly.
2. Train with gradient descent to minimize MSE.
3. Compare the learned projection $W_d W_e$ with the PCA projection $DD^\top$.

In [ ]:
# Exercise 11: Linear autoencoder = PCA

# Settings
l_ae = 1          # bottleneck dimension (keep 1 component)
lr = 5e-4         # learning rate
n_epochs = 8000

rng_ae = np.random.default_rng(123)

# Initialize weights
We = rng_ae.standard_normal((l_ae, 2)) * 0.1   # encoder: (1, 2)
Wd = rng_ae.standard_normal((2, l_ae)) * 0.1   # decoder: (2, 1)

losses = []

for epoch in range(n_epochs):
    # Forward pass
    C_ae = X @ We.T          # (n, l_ae) — codes
    X_recon_ae = C_ae @ Wd.T # (n, 2)   — reconstructions
    residuals = X - X_recon_ae
    loss = np.mean(np.sum(residuals**2, axis=1))
    losses.append(loss)

    # Backward pass (manual gradients)
    # d(Loss)/d(Wd) = -2/n * residuals.T @ C_ae -> shape (2, l_ae)
    grad_Wd = -2.0 / n * residuals.T @ C_ae
    # d(Loss)/d(We) = -2/n * (Wd.T @ residuals.T) @ X -> shape (l_ae, 2)
    grad_We = -2.0 / n * (residuals.T @ X).T @ Wd  # careful with shapes
    grad_We = grad_We.T  # (l_ae, 2)

    # Gradient descent update
    Wd -= lr * grad_Wd
    We -= lr * grad_We

# Learned projection matrix
P_ae = Wd @ We   # (2, 2) — should approximate D @ D.T

# PCA projection matrix (rank 1)
U_ae, s_ae, Vt_ae = np.linalg.svd(X, full_matrices=False)
D_pca = Vt_ae[0:1].T  # (2, 1)
P_pca = D_pca @ D_pca.T  # (2, 2)


# ── Verification ──
# The projection matrices should be close
proj_error = np.linalg.norm(P_ae - P_pca, 'fro')
print(f"Autoencoder projection matrix:\n{np.round(P_ae, 4)}")
print(f"\nPCA projection matrix (D @ D.T):\n{np.round(P_pca, 4)}")
print(f"\nFrobenius norm of difference: {proj_error:.6f}")
assert proj_error < 0.05, f"Projection matrices differ too much: {proj_error:.4f}"
print(f"Final loss: {losses[-1]:.4f}")
print("\nExercise 11: PASSED")

# ── Plot training loss ──
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(losses, lw=1)
ax.axhline(losses[-1], color='red', ls='--', alpha=0.5,
           label=f'Final loss = {losses[-1]:.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Linear Autoencoder Training (converges to PCA solution)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 12 — Noise Reduction with PCA  *(intermediate)*

One practical use of PCA is **denoising**: add noise to data, then reconstruct
using only the top components. Since noise spreads across all components equally
while signal concentrates in the top components, discarding the bottom ones
removes noise more than signal.

**Tasks:**
1. Add Gaussian noise to the data.
2. Compute PCA of the noisy data.
3. Reconstruct using only the top component.
4. Measure how close the denoised data is to the original clean data.

In [ ]:
# Exercise 12: Noise reduction with PCA

rng_noise = np.random.default_rng(99)
noise_level = 1.0
X_noisy = X + rng_noise.standard_normal(X.shape) * noise_level
X_noisy = X_noisy - X_noisy.mean(axis=0)  # re-center

# PCA of noisy data
U_n, s_n, Vt_n = np.linalg.svd(X_noisy, full_matrices=False)

# Reconstruct using only PC1
pc1_noisy = Vt_n[0]
scores_noisy = X_noisy @ pc1_noisy
X_denoised = np.outer(scores_noisy, pc1_noisy)

# Also re-center X for fair comparison
X_centered = X - X.mean(axis=0)

# Measure errors
err_noisy    = np.mean(np.sum((X_centered - X_noisy)**2, axis=1))
err_denoised = np.mean(np.sum((X_centered - X_denoised)**2, axis=1))

print(f"Mean squared error (noisy vs clean):    {err_noisy:.4f}")
print(f"Mean squared error (denoised vs clean): {err_denoised:.4f}")
print(f"Error reduction: {(1 - err_denoised / err_noisy) * 100:.1f}%")

assert err_denoised < err_noisy, "Denoising should reduce error!"
print("\nExercise 12: PASSED")

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, data, title in zip(axes,
        [X_centered, X_noisy, X_denoised],
        ['Clean data', 'Noisy data', 'Denoised (PC1 only)']):
    ax.scatter(data[:, 0], data[:, 1], alpha=0.4, s=15)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-8, 8)
    ax.set_ylim(-6, 6)

plt.tight_layout()
plt.show()

---
## Summary

This lab derived PCA from first principles using every major concept from
Chapter 2 of the Deep Learning book:

| Concept (Lab) | Role in PCA |
|---------------|-------------|
| **Vectors, matrices** (Lab 01) | Data is a matrix $X$, components are vectors |
| **Matrix multiply** (Lab 01) | Encoding $c = D^\top x$, decoding $\hat{x} = Dc$ |
| **Orthogonality** (Lab 01, 03) | $D^\top D = I$ constraint; Gram-Schmidt |
| **Linear systems / Projection** (Lab 02) | $DD^\top$ is an orthogonal projection matrix |
| **Norms** (Lab 03) | Reconstruction error measured by $\|\cdot\|_2^2$ |
| **Eigendecomposition** (Lab 04) | $C = Q\Lambda Q^\top$ gives principal directions |
| **Spectral theorem** (Lab 04) | Guarantees real, orthogonal eigenvectors for symmetric $C$ |
| **SVD** (Lab 05) | $X = U\Sigma V^\top$ gives PCA without forming $X^\top X$ |
| **Trace** (Lab 05) | Connects Frobenius norm to the optimization objective |

### Key formulas to remember

$$\boxed{\text{Encode: } \mathbf{c} = D^\top \mathbf{x}}$$

$$\boxed{\text{Decode: } \hat{\mathbf{x}} = D\mathbf{c} = DD^\top\mathbf{x}}$$

$$\boxed{\text{Optimal } D = \text{top } l \text{ eigenvectors of } X^\top X}$$

$$\boxed{\text{Reconstruction error} = \sum_{j=l+1}^{n} \lambda_j}$$

$$\boxed{\text{Variance explained by component } k = \frac{\lambda_k}{\sum_j \lambda_j}}$$

### What comes next

With the linear algebra foundation complete, the natural next steps are:
- **Probability and information theory** (Chapter 3): needed for probabilistic PCA
  and understanding why maximum variance = maximum information for Gaussian data
- **Optimization** (Chapter 4): the gradient-based derivation we did here extends
  to nonlinear autoencoders
- **Deep feedforward networks** (Chapter 6): where linear autoencoders become
  *nonlinear* autoencoders, generalizing PCA to curved manifolds